In [37]:
"""
seed_rental_db.py (FIXED)
========================
Generates realistic synthetic data for the Rental Analytics project.

Fixes applied:
  1. No overlapping rental dates per product
  2. No returned-before-start (clamp return >= start + 1)
  3. Overdue capped at 90 days -> not_returned status
  4. Customer stats backfilled from actual rentals
  5. Added join_date and still_active to customers
  6. Removed email column from customers
"""

import os
import random
import math
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker
fake = Faker("pt_PT")

random.seed(42)
np.random.seed(42)

In [38]:
# -- Output paths --
DATA_DIR  = Path("generated_data")
DATA_DIR.mkdir(exist_ok=True)

# -- Simulation window --
SIM_START = date(2022, 1, 1)
SIM_END   = date(2024, 12, 31)
TODAY     = date(2025, 4, 6)

# -- Overdue cap --
MAX_OVERDUE_DAYS = 90

def rand_date(start: date, end: date) -> date:
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, max(delta, 0)))

def days_between(d1: date, d2: date) -> int:
    return (d2 - d1).days

In [39]:
# CATEGORY DEFINITIONS
CATEGORIES = [
    {
        "name": "Televisions",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 210,
        "sale_days_sigma": 0.9,
        "n_products": 280,
        "price_range": (299, 2499),
        "brands": ["Samsung", "LG", "Sony", "Philips", "TCL", "Hisense"],
        "rental_demand_mu": 3.2,
        "rental_duration_mu": 14,
        "rental_duration_sigma": 5,
    },
    {
        "name": "Drones",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 280,
        "sale_days_sigma": 1.0,
        "n_products": 180,
        "price_range": (149, 1899),
        "brands": ["DJI", "Parrot", "Autel", "Holy Stone", "Ruko"],
        "rental_demand_mu": 4.5,
        "rental_duration_mu": 5,
        "rental_duration_sigma": 2,
    },
    {
        "name": "Gaming Consoles",
        "segment": "gaming",
        "avg_days_to_sale": 120,
        "sale_days_sigma": 0.7,
        "n_products": 200,
        "price_range": (249, 699),
        "brands": ["Sony", "Microsoft", "Nintendo"],
        "rental_demand_mu": 6.0,
        "rental_duration_mu": 10,
        "rental_duration_sigma": 4,
    },
    {
        "name": "Laptops",
        "segment": "computing",
        "avg_days_to_sale": 190,
        "sale_days_sigma": 0.85,
        "n_products": 320,
        "price_range": (399, 2999),
        "brands": ["Apple", "Dell", "HP", "Lenovo", "Asus", "Acer", "MSI"],
        "rental_demand_mu": 5.5,
        "rental_duration_mu": 21,
        "rental_duration_sigma": 7,
    },
    {
        "name": "Cameras & Photography",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 240,
        "sale_days_sigma": 0.95,
        "n_products": 220,
        "price_range": (199, 3499),
        "brands": ["Canon", "Nikon", "Sony", "Fujifilm", "Panasonic", "Olympus"],
        "rental_demand_mu": 5.0,
        "rental_duration_mu": 7,
        "rental_duration_sigma": 3,
    },
    {
        "name": "Smart Home & Audio",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 160,
        "sale_days_sigma": 0.75,
        "n_products": 260,
        "price_range": (49, 899),
        "brands": ["Sonos", "Bose", "Amazon", "Google", "Apple", "JBL", "Bang & Olufsen"],
        "rental_demand_mu": 2.8,
        "rental_duration_mu": 30,
        "rental_duration_sigma": 10,
    },
    {
        "name": "Keyboards & Peripherals",
        "segment": "computing",
        "avg_days_to_sale": 130,
        "sale_days_sigma": 0.65,
        "n_products": 240,
        "price_range": (29, 399),
        "brands": ["Logitech", "Razer", "Corsair", "SteelSeries", "Keychron", "Das Keyboard"],
        "rental_demand_mu": 2.0,
        "rental_duration_mu": 14,
        "rental_duration_sigma": 5,
    },
    {
        "name": "Projectors",
        "segment": "consumer_electronics",
        "avg_days_to_sale": 300,
        "sale_days_sigma": 1.05,
        "n_products": 180,
        "price_range": (199, 4999),
        "brands": ["Epson", "Optoma", "BenQ", "LG", "Sony", "Anker"],
        "rental_demand_mu": 6.5,
        "rental_duration_mu": 3,
        "rental_duration_sigma": 1,
    },
]

In [40]:
# Adjectives for product name generation
ADJECTIVES = [
    "Ultra", "Pro", "Max", "Plus", "Elite", "Smart", "Advanced",
    "Premium", "Slim", "Compact", "Wireless", "4K", "HD", "Mini",
    "Neo", "Air", "Edge", "Flex", "Vision",
]
TV_SIZES    = [43, 50, 55, 65, 75, 85]
TV_TYPES    = ["QLED", "OLED", "LED", "Neo QLED", "MiniLED", "AMOLED"]
DRONE_TYPES = ["Nano", "Mini 3", "Mini 4", "Air 3", "Pro", "Enterprise"]
CONSOLE_GEN = {
    "Sony": ["PlayStation 5", "PlayStation 5 Digital", "PlayStation 4 Pro"],
    "Microsoft": ["Xbox Series X", "Xbox Series S", "Xbox One X"],
    "Nintendo": ["Switch OLED", "Switch", "Switch Lite"],
}

def make_product_name(cat_name: str, brand: str) -> str:
    adj  = random.choice(ADJECTIVES)
    year = random.choice([2021, 2022, 2023, 2024])
    model_num = f"{random.randint(1,9)}{''.join([str(random.randint(0,9)) for _ in range(2)])}"

    if cat_name == "Televisions":
        sz = random.choice(TV_SIZES)
        tp = random.choice(TV_TYPES)
        return f"{brand} {sz}\" {tp} {adj} {year}"
    elif cat_name == "Drones":
        tp = random.choice(DRONE_TYPES)
        return f"{brand} {tp} {adj}"
    elif cat_name == "Gaming Consoles":
        options = CONSOLE_GEN.get(brand, [f"{brand} Console"])
        return random.choice(options)
    elif cat_name == "Laptops":
        return f"{brand} {adj} {model_num} ({year})"
    elif cat_name == "Cameras & Photography":
        return f"{brand} {adj} {model_num} {'Mirrorless' if random.random() > 0.4 else 'DSLR'}"
    elif cat_name == "Smart Home & Audio":
        tp = random.choice(["Speaker", "Soundbar", "Smart Display", "Smart Hub", "Subwoofer"])
        return f"{brand} {adj} {tp}"
    elif cat_name == "Keyboards & Peripherals":
        tp = random.choice(["Mechanical Keyboard", "Gaming Mouse", "Wireless Combo",
                            "USB-C Hub", "Webcam", "Headset"])
        return f"{brand} {adj} {tp}"
    elif cat_name == "Projectors":
        return f"{brand} {adj} {model_num} {'4K' if random.random() > 0.5 else 'FHD'} Projector"
    return f"{brand} {adj} {model_num}"

In [41]:
# BUILD TABLES

def build_categories() -> pd.DataFrame:
    rows = []
    for i, c in enumerate(CATEGORIES, start=1):
        rows.append({
            "category_id":      i,
            "name":             c["name"],
            "segment":          c["segment"],
            "avg_days_to_sale": c["avg_days_to_sale"],
        })
    return pd.DataFrame(rows)


def build_products(cats_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    sku_counter = 10000
    conditions  = ["new"] * 80 + ["open_box"] * 15 + ["refurbished"] * 5

    for cat_idx, cat in enumerate(CATEGORIES, start=1):
        for _ in range(cat["n_products"]):
            brand = random.choice(cat["brands"])
            name  = make_product_name(cat["name"], brand)

            lo, hi = cat["price_range"]
            retail_price = round(random.uniform(lo, hi) / 5) * 5

            listed_date = rand_date(SIM_START, SIM_END - timedelta(days=30))

            mu_days = cat["avg_days_to_sale"]
            sigma   = cat["sale_days_sigma"]
            ln_mu   = math.log(mu_days) - (sigma**2) / 2
            days_to_sale = int(np.random.lognormal(ln_mu, sigma))
            days_to_sale = max(30, min(days_to_sale, 1200))

            sell_date     = listed_date + timedelta(days=days_to_sale)
            days_on_shelf = days_between(listed_date, TODAY)

            if sell_date <= TODAY:
                status = "sold"
            elif days_on_shelf >= 365:
                status = random.choices(
                    ["eligible_for_rental", "rented", "eligible_for_rental"],
                    weights=[50, 35, 15]
                )[0]
            else:
                status = "for_sale"

            rows.append({
                "product_id":       len(rows) + 1,
                "category_id":      cat_idx,
                "sku":              f"SKU-{sku_counter}",
                "name":             name,
                "brand":            brand,
                "retail_price_eur": retail_price,
                "release_date":     (listed_date - timedelta(days=random.randint(0, 90))).isoformat(),
                "listed_date":      listed_date.isoformat(),
                "condition":        random.choice(conditions),
                "status":           status,
                "days_on_shelf":    days_on_shelf,
            })
            sku_counter += random.randint(1, 9)

    return pd.DataFrame(rows)

In [42]:
def build_inventory_events(products_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    event_id = 1

    for _, p in products_df.iterrows():
        listed        = date.fromisoformat(p["listed_date"])
        price         = float(p["retail_price_eur"])
        pid           = int(p["product_id"])
        status        = p["status"]
        days_on_shelf = int(p["days_on_shelf"])

        # Always: listed event
        rows.append({
            "event_id":       event_id,
            "product_id":     pid,
            "event_type":     "listed",
            "event_date":     listed.isoformat(),
            "price_at_event": price,
            "notes":          "Initial stock entry",
        })
        event_id += 1

        # Price reduction: only if item has been on shelf long enough
        reduction_window = min(days_on_shelf - 30, 360)
        if days_on_shelf > 210 and reduction_window > 180 and random.random() < 0.4:
            reduction_date = listed + timedelta(days=random.randint(180, reduction_window))
            reduced_price  = round(price * random.uniform(0.70, 0.90), 2)
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "price_reduced",
                "event_date":     reduction_date.isoformat(),
                "price_at_event": reduced_price,
                "notes":          f"Seasonal markdown to {reduced_price}E",
            })
            event_id += 1

        # Rental eligible: only if item has been on shelf >= 365 days
        if days_on_shelf >= 365:
            eligible_date = listed + timedelta(days=365)
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "rental_eligible",
                "event_date":     eligible_date.isoformat(),
                "price_at_event": price,
                "notes":          "Crossed 12-month threshold -- eligible for rental program",
            })
            event_id += 1

            # FIX: rented_out event must happen AFTER rental_eligible (day 365+)
            if status == "rented":
                days_after_eligible = random.randint(1, 60)
                rented_out_date = listed + timedelta(days=365 + days_after_eligible)
                # Safety: must not be in the future
                if rented_out_date <= TODAY:
                    rows.append({
                        "event_id":       event_id,
                        "product_id":     pid,
                        "event_type":     "rented_out",
                        "event_date":     rented_out_date.isoformat(),
                        "price_at_event": price,
                        "notes":          "Assigned to rental program",
                    })
                    event_id += 1

        # FIX: sold event uses actual sell_date, not TODAY
        if status == "sold":
            # Reconstruct sell_date from the product's log-normal draw
            # We don't have it stored, so use listed + days_on_shelf as upper bound
            # and pick a random plausible sell date before TODAY
            latest_possible = min(
                listed + timedelta(days=days_on_shelf),
                TODAY - timedelta(days=1)
            )
            earliest_possible = listed + timedelta(days=30)
            if earliest_possible < latest_possible:
                sell_date = rand_date(earliest_possible, latest_possible)
            else:
                sell_date = latest_possible
            rows.append({
                "event_id":       event_id,
                "product_id":     pid,
                "event_type":     "sold",
                "event_date":     sell_date.isoformat(),
                "price_at_event": round(price * random.uniform(0.55, 1.0), 2),
                "notes":          "Final sale",
            })
            event_id += 1

    return pd.DataFrame(rows)

In [43]:
def build_pricing_rules(cats_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    rate_map = {
        "Televisions":          0.0045,
        "Drones":               0.0060,
        "Gaming Consoles":      0.0055,
        "Laptops":              0.0040,
        "Cameras & Photography":0.0065,
        "Smart Home & Audio":   0.0035,
        "Keyboards & Peripherals": 0.0030,
        "Projectors":           0.0070,
    }
    deposit_map = {
        "Televisions":          0.25,
        "Drones":               0.30,
        "Gaming Consoles":      0.20,
        "Laptops":              0.25,
        "Cameras & Photography":0.30,
        "Smart Home & Audio":   0.15,
        "Keyboards & Peripherals": 0.15,
        "Projectors":           0.25,
    }
    for _, cat in cats_df.iterrows():
        cname = cat["name"]
        rows.append({
            "rule_id":              int(cat["category_id"]),
            "category_id":          int(cat["category_id"]),
            "age_min_days":         365,
            "age_max_days":         None,
            "base_daily_rate_pct":  rate_map.get(cname, 0.005),
            "deposit_pct":          deposit_map.get(cname, 0.20),
            "late_fee_daily":       random.choice([3.0, 5.0, 7.5, 10.0]),
            "active":               1,
        })
    return pd.DataFrame(rows)

In [44]:
# FIX #5 + #6: Customers -- no email, placeholder stats (backfilled later)
def build_customers(n: int = 800) -> pd.DataFrame:
    rows = []
    for i in range(1, n + 1):
        nif_digits = "".join([str(random.randint(0, 9)) for _ in range(9)])

        rows.append({
            "customer_id":      i,
            "full_name":        fake.name(),
            # email removed (Bug #6)
            "nif":              nif_digits,
            "phone":            fake.phone_number(),
            "address":          fake.address().replace("\n", ", "),
            "id_verified":      1 if random.random() > 0.1 else 0,
            # placeholders -- backfilled from actual rentals (Bug #4)
            "total_rentals":    0,
            "late_returns":     0,
            "avg_return_days":  None,
            "churn_risk_score": 0.0,
            "join_date":        None,      # Bug #5: set after rentals
            "still_active":     1,          # Bug #5: set after backfill
        })
    return pd.DataFrame(rows)

In [45]:
# RENTALS -- with all fixes applied
# FIX #1: No overlapping dates per product (track last_end per product)
# FIX #2: Clamp return date >= start + 1 day
# FIX #3: Cap overdue at MAX_OVERDUE_DAYS, not_returned status
def build_rentals(products_df: pd.DataFrame,
                  customers_df: pd.DataFrame,
                  pricing_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    rentable = products_df[
        products_df["status"].isin(["eligible_for_rental", "rented"]) |
        (products_df["days_on_shelf"] >= 365)
    ].copy()

    price_lookup = {
        int(r["category_id"]): r
        for _, r in pricing_df.iterrows()
    }

    rental_rows    = []
    condition_rows = []
    rental_id      = 1
    condition_id   = 1

    cat_demand   = {c["name"]: c["rental_demand_mu"] for c in CATEGORIES}
    cat_dur      = {c["name"]: (c["rental_duration_mu"], c["rental_duration_sigma"]) for c in CATEGORIES}
    cat_name_map = {i+1: c["name"] for i, c in enumerate(CATEGORIES)}

    # FIX #1: Track per-product last rental end date
    product_last_end = {}

    sim_months = pd.date_range(SIM_START.isoformat(), SIM_END.isoformat(), freq="MS")

    for month_start in sim_months:
        month_date = month_start.date()

        for _, product in rentable.sample(frac=0.15, random_state=random.randint(0, 9999)).iterrows():
            cat_id   = int(product["category_id"])
            cat_name = cat_name_map.get(cat_id, "Unknown")
            rule     = price_lookup.get(cat_id)
            if rule is None:
                continue

            retail     = float(product["retail_price_eur"])
            daily_rate = round(retail * float(rule["base_daily_rate_pct"]), 2)
            deposit    = round(retail * float(rule["deposit_pct"]), 2)

            demand_mu  = cat_demand.get(cat_name, 3.0)
            n_rentals  = max(0, int(np.random.poisson(demand_mu / 10)))

            for _ in range(n_rentals):
                if rental_id > 3500:
                    break

                dur_mu, dur_sigma = cat_dur.get(cat_name, (10, 4))
                duration = max(1, int(np.random.normal(dur_mu, dur_sigma)))

                pid = int(product["product_id"])
                listed_date = date.fromisoformat(product["listed_date"])
                rental_start = month_date + timedelta(days=random.randint(0, 27))

                # FIX #7: Rental must happen after product was listed
                if rental_start < listed_date:
                    continue

                # FIX #1: Shift start if it overlaps previous rental
                last_end = product_last_end.get(pid)
                if last_end and rental_start <= last_end:
                    rental_start = last_end + timedelta(days=1)

                rental_due = rental_start + timedelta(days=duration)

                is_past = rental_due < TODAY
                if is_past:
                    outcome = random.choices(
                        ["on_time", "late", "still_out"],
                        weights=[75, 17, 8]
                    )[0]
                else:
                    outcome = "still_out"

                late_days = 0
                if outcome == "on_time":
                    # FIX #2: Clamp return >= start + 1 day
                    rental_returned = max(
                        rental_start + timedelta(days=1),
                        rental_due - timedelta(days=random.randint(0, 2))
                    )
                    status = "returned"
                elif outcome == "late":
                    late_days = random.randint(1, 14)
                    rental_returned = rental_due + timedelta(days=late_days)
                    status = "returned"
                else:
                    rental_returned = None
                    # FIX #3: Cap overdue at MAX_OVERDUE_DAYS
                    raw_days = days_between(rental_start, TODAY)
                    if raw_days > MAX_OVERDUE_DAYS:
                        status = "not_returned"
                    else:
                        status = "active" if rental_due >= TODAY else "overdue"

                # Calculate days_held
                if rental_returned:
                    days_held = days_between(rental_start, rental_returned)
                elif status == "not_returned":
                    days_held = MAX_OVERDUE_DAYS  # FIX #3: capped
                else:
                    days_held = days_between(rental_start, TODAY)

                # Calculate total_charged
                if status == "not_returned":
                    # FIX #3: full retail price + late fees for 90 days
                    late_fee_total = float(rule["late_fee_daily"]) * MAX_OVERDUE_DAYS
                    total_charged = round(retail + late_fee_total, 2)
                else:
                    late_fee = float(rule["late_fee_daily"]) * late_days
                    total_charged = round(daily_rate * days_held + late_fee, 2)

                customer = customers_df.sample(1).iloc[0]

                # FIX #1: Update product last end date
                # FIX #2b: For unreturned rentals, block future rentals with far-future date
                if rental_returned:
                    product_last_end[pid] = rental_returned
                elif status in ("not_returned", "overdue", "active"):
                    product_last_end[pid] = date(2099, 12, 31)  # product is still out
                else:
                    product_last_end[pid] = rental_due

                rental_rows.append({
                    "rental_id":       rental_id,
                    "product_id":      pid,
                    "customer_id":     int(customer["customer_id"]),
                    "rental_start":    rental_start.isoformat(),
                    "rental_due":      rental_due.isoformat(),
                    "rental_returned": rental_returned.isoformat() if rental_returned else None,
                    "daily_rate_eur":  daily_rate,
                    "deposit_eur":     deposit,
                    "days_held":       days_held,
                    "total_charged":   total_charged,
                    "status":          status,
                })

                if rental_returned is not None:
                    cond = random.choices(
                        ["pristine", "good", "fair", "damaged", "destroyed"],
                        weights=[30, 45, 18, 6, 1]
                    )[0]
                    damage = 0.0
                    dep_returned = True
                    if cond == "damaged":
                        damage = round(random.uniform(20, retail * 0.3), 2)
                        dep_returned = random.random() > 0.5
                    elif cond == "destroyed":
                        damage = round(retail * random.uniform(0.5, 1.0), 2)
                        dep_returned = False

                    condition_rows.append({
                        "condition_id":        condition_id,
                        "rental_id":           rental_id,
                        "assessed_condition":  cond,
                        "damage_charge":       damage,
                        "deposit_returned":    1 if dep_returned else 0,
                        "inspector_notes":     f"Assessed after return on {rental_returned.isoformat()}",
                    })
                    condition_id += 1

                rental_id += 1

    return pd.DataFrame(rental_rows), pd.DataFrame(condition_rows)

In [46]:
# FIX #4 + #5: Backfill customer stats from actual rentals
def backfill_customer_stats(customers_df: pd.DataFrame,
                            rentals_df: pd.DataFrame) -> pd.DataFrame:
    """
    Replaces placeholder customer stats with values derived from actual rentals.
    Also sets join_date (1-60 days before first rental) and still_active.
    """
    # Compute per-customer aggregates from real rentals
    agg = rentals_df.groupby("customer_id").agg(
        total_rentals=("rental_id", "count"),
        avg_return_days=("days_held", "mean"),
        first_rental=("rental_start", "min"),
    ).reset_index()

    # Count late returns: returned after due date
    late_df = rentals_df[
        (rentals_df["rental_returned"].notna()) &
        (rentals_df["rental_returned"] > rentals_df["rental_due"])
    ].groupby("customer_id").size().reset_index(name="late_returns")

    agg = agg.merge(late_df, on="customer_id", how="left")
    agg["late_returns"] = agg["late_returns"].fillna(0).astype(int)
    agg["avg_return_days"] = agg["avg_return_days"].round(1)

    # Drop placeholder columns from customers
    drop_cols = ["total_rentals", "late_returns", "avg_return_days",
                 "churn_risk_score", "join_date", "still_active"]
    cust = customers_df.drop(columns=drop_cols)

    # Merge real stats
    cust = cust.merge(agg, on="customer_id", how="left")
    cust["total_rentals"]   = cust["total_rentals"].fillna(0).astype(int)
    cust["late_returns"]    = cust["late_returns"].fillna(0).astype(int)
    cust["avg_return_days"] = cust["avg_return_days"].where(cust["total_rentals"] > 0, None)

    # Churn risk from real data
    late_ratio = cust["late_returns"] / cust["total_rentals"].clip(lower=1)
    cust["churn_risk_score"] = (
        late_ratio * 0.6 + np.random.uniform(0, 0.4, len(cust))
    ).clip(0, 1).round(3)

    # FIX #5: join_date = 1-60 days before first rental
    cust["first_rental"] = pd.to_datetime(cust["first_rental"])
    offsets = np.random.randint(1, 61, size=len(cust))
    cust["join_date"] = cust["first_rental"] - pd.to_timedelta(offsets, unit="D")
    # Customers with no rentals: random join date in simulation window
    no_rental_mask = cust["first_rental"].isna()
    n_missing = no_rental_mask.sum()
    if n_missing > 0:
        random_days = [random.randint(0, (SIM_END - SIM_START).days) for _ in range(n_missing)]
        cust.loc[no_rental_mask, "join_date"] = [
            pd.Timestamp(SIM_START + timedelta(days=d)) for d in random_days
        ]
    cust["join_date"] = cust["join_date"].dt.date.astype(str)
    cust.drop(columns=["first_rental"], inplace=True)

    # FIX #5: still_active -- inactive if high churn + many late returns
    cust["still_active"] = (~(
        (cust["churn_risk_score"] > 0.8) & (cust["late_returns"] > 3)
    )).astype(int)

    return cust

In [47]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM v_inventory_aging", engine)

In [48]:
# WRITE TO MYSQL + CSV

def write_all():
    print("Building tables...")

    from sqlalchemy import create_engine, text
    from dotenv import load_dotenv
    import os

    load_dotenv()

    engine = create_engine(
        f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
        f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
    )

    cats_df      = build_categories()
    products_df  = build_products(cats_df)
    events_df    = build_inventory_events(products_df)
    pricing_df   = build_pricing_rules(cats_df)
    customers_df = build_customers(800)
    rentals_df, conditions_df = build_rentals(products_df, customers_df, pricing_df)

    # FIX #4 + #5: Backfill customer stats from actual rental data
    customers_df = backfill_customer_stats(customers_df, rentals_df)

    # FIX #8: Sync product status with actual rentals
    # Products marked "rented" must have at least one active/overdue/not_returned rental
    active_rental_pids = set(
        rentals_df[rentals_df["status"].isin(["active", "overdue", "not_returned"])]["product_id"].unique()
    )
    products_df["status"] = products_df.apply(
        lambda row: "rented" if int(row["product_id"]) in active_rental_pids
        else ("eligible_for_rental" if row["status"] == "rented" else row["status"]),
        axis=1
    )

    tables = {
        "categories":        cats_df,
        "products":          products_df,
        "inventory_events":  events_df,
        "pricing_rules":     pricing_df,
        "customers":         customers_df,
        "rentals":           rentals_df,
        "return_conditions": conditions_df,
    }

    with engine.connect() as con:
        con.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
        for table_name in tables.keys():
            con.execute(text("DROP TABLE IF EXISTS " + table_name))
        con.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
        con.commit()

    for table_name, df in tables.items():
        df.to_sql(table_name, engine, if_exists="replace", index=False)
        print(f"  {table_name:25s}  {len(df):>6,} rows")

    # -- Analytical views --
    views = [
        "DROP VIEW IF EXISTS v_rental_eligible",
        """CREATE VIEW v_rental_eligible AS
        SELECT
            p.product_id, p.sku, p.name, p.brand,
            c.name AS category, c.segment,
            p.retail_price_eur, p.listed_date,
            DATEDIFF(CURDATE(), p.listed_date) AS days_on_shelf,
            p.condition, pr.base_daily_rate_pct,
            ROUND(p.retail_price_eur * pr.base_daily_rate_pct, 2)      AS daily_rate_eur,
            ROUND(p.retail_price_eur * pr.deposit_pct, 2)              AS deposit_eur,
            ROUND(p.retail_price_eur * 0.60, 2)                        AS markdown_revenue,
            ROUND(p.retail_price_eur * pr.base_daily_rate_pct * 90, 2) AS rental_90d_revenue
        FROM products p
        JOIN categories c     ON p.category_id = c.category_id
        JOIN pricing_rules pr ON pr.category_id = c.category_id
        WHERE p.status IN ('eligible_for_rental', 'rented') AND pr.active = 1""",

        "DROP VIEW IF EXISTS v_rental_history",
        """CREATE VIEW v_rental_history AS
        SELECT
            r.rental_id,
            p.name AS product_name, c.name AS category, cu.full_name AS customer_name,
            r.rental_start, r.rental_due, r.rental_returned,
            DATEDIFF(COALESCE(r.rental_returned, CURDATE()), r.rental_start) AS days_held,
            r.daily_rate_eur, r.deposit_eur, r.total_charged, r.status,
            rc.assessed_condition, rc.damage_charge, rc.deposit_returned,
            CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END AS returned_late,
            DATEDIFF(COALESCE(r.rental_returned, CURDATE()), r.rental_due) AS days_overdue
        FROM rentals r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        JOIN customers cu ON r.customer_id = cu.customer_id
        LEFT JOIN return_conditions rc ON r.rental_id = rc.rental_id""",

        "DROP VIEW IF EXISTS v_inventory_aging",
        """CREATE VIEW v_inventory_aging AS
        SELECT
            c.name AS category, c.segment,
            COUNT(*) AS total_items,
            ROUND(AVG(DATEDIFF(CURDATE(), p.listed_date)), 1) AS avg_days_on_shelf,
            SUM(p.retail_price_eur) AS total_retail_value_eur,
            SUM(CASE WHEN DATEDIFF(CURDATE(), p.listed_date) > 365
                     THEN p.retail_price_eur ELSE 0 END) AS stale_inventory_value_eur,
            COUNT(CASE WHEN p.status = 'eligible_for_rental' THEN 1 END) AS rental_eligible_count,
            COUNT(CASE WHEN p.status = 'sold' THEN 1 END) AS sold_count
        FROM products p
        JOIN categories c ON p.category_id = c.category_id
        GROUP BY c.name, c.segment""",

        "DROP VIEW IF EXISTS v_revenue_comparison",
        """CREATE VIEW v_revenue_comparison AS
        SELECT
            c.name AS category,
            COUNT(DISTINCT r.rental_id) AS total_rentals,
            ROUND(SUM(r.total_charged), 2) AS total_rental_revenue,
            ROUND(AVG(r.total_charged), 2) AS avg_revenue_per_rental,
            ROUND(AVG(r.daily_rate_eur), 2) AS avg_daily_rate,
            ROUND(AVG(r.days_held), 1) AS avg_days_held,
            SUM(CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END) AS late_returns,
            ROUND(
                100.0 * SUM(CASE WHEN r.rental_returned > r.rental_due THEN 1 ELSE 0 END)
                / NULLIF(COUNT(r.rental_id), 0), 1
            ) AS late_return_pct
        FROM rentals r
        JOIN products p   ON r.product_id  = p.product_id
        JOIN categories c ON p.category_id = c.category_id
        GROUP BY c.name""",
    ]

    with engine.connect() as con:
        for statement in views:
            con.execute(text(statement))
        con.commit()

    # -- CSVs --
    CSV_DIR = Path("../data/raw")
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    for table_name, df in tables.items():
        out_path = CSV_DIR / f"{table_name}.csv"
        if not out_path.exists():
            df.to_csv(out_path, index=False)
            print(f"  Written: {out_path}")
        else:
            print(f"  Skipped (already exists): {out_path}")
            
    # -- Summary --
    print()
    print("=" * 55)
    print("  Data generation complete")
    print("=" * 55)

    summary = pd.read_sql("""
        SELECT category, total_items, avg_days_on_shelf,
               ROUND(stale_inventory_value_eur, 0) AS stale_eur,
               rental_eligible_count
        FROM v_inventory_aging
        ORDER BY stale_eur DESC
    """, engine)
    print("  Inventory aging summary:")
    print(summary.to_string(index=False))

    rev = pd.read_sql("""
        SELECT category, total_rentals,
               ROUND(total_rental_revenue, 0) AS rental_revenue_eur,
               avg_days_held, late_return_pct
        FROM v_revenue_comparison
        ORDER BY rental_revenue_eur DESC
    """, engine)
    print("\n  Revenue by category:")
    print(rev.to_string(index=False))

    headline = pd.read_sql("""
        SELECT
            ROUND(SUM(rental_90d_revenue), 0) AS rental_potential_eur,
            ROUND(SUM(markdown_revenue), 0)   AS markdown_potential_eur,
            ROUND(
                100.0 * (SUM(rental_90d_revenue) - SUM(markdown_revenue))
                / SUM(markdown_revenue), 1
            ) AS rental_upside_pct
        FROM v_rental_eligible
    """, engine)
    print("\n  Core hypothesis (eligible inventory only):")
    print(f"  Rental potential (90d):   E{int(headline['rental_potential_eur'].iloc[0]):,}")
    print(f"  Markdown alternative:     E{int(headline['markdown_potential_eur'].iloc[0]):,}")
    print(f"  Rental upside:            {headline['rental_upside_pct'].iloc[0]}%")

    # -- Integrity checks --
    print("\n  Integrity checks:")
    overlap_check = pd.read_sql("""
        SELECT COUNT(*) AS overlaps FROM (
            SELECT r1.rental_id
            FROM rentals r1
            JOIN rentals r2 ON r1.product_id = r2.product_id
                AND r1.rental_id < r2.rental_id
                AND r1.rental_start < COALESCE(r2.rental_returned, r2.rental_due)
                AND r2.rental_start < COALESCE(r1.rental_returned, r1.rental_due)
        ) t
    """, engine)
    print(f"  Overlapping rentals:      {int(overlap_check['overlaps'].iloc[0])}")

    neg_check = pd.read_sql("SELECT COUNT(*) AS n FROM rentals WHERE days_held < 1", engine)
    print(f"  Negative/zero days_held:  {int(neg_check['n'].iloc[0])}")

    overdue_check = pd.read_sql(
        f"SELECT COUNT(*) AS n FROM rentals WHERE days_held > {MAX_OVERDUE_DAYS} AND rental_returned IS NULL",
        engine
    )
    print(f"  Overdue > {MAX_OVERDUE_DAYS} days:         {int(overdue_check['n'].iloc[0])}")

    print("\n  Done. Open MySQL Workbench to explore your data.")


write_all()

Building tables...
  categories                      8 rows
  products                    1,880 rows
  inventory_events            5,801 rows
  pricing_rules                   8 rows
  customers                     800 rows
  rentals                     2,204 rows
  return_conditions           1,860 rows
  Written: ..\data\raw\categories.csv
  Written: ..\data\raw\products.csv
  Written: ..\data\raw\inventory_events.csv
  Written: ..\data\raw\pricing_rules.csv
  Written: ..\data\raw\customers.csv
  Written: ..\data\raw\rentals.csv
  Written: ..\data\raw\return_conditions.csv

  Data generation complete
  Inventory aging summary:
               category  total_items  avg_days_on_shelf  stale_eur  rental_eligible_count
                Laptops          320             1054.6   540575.0                      7
             Projectors          180             1030.8   491945.0                     17
  Cameras & Photography          220             1016.9   405635.0                     11
   